# 在 `hcnote/Cybersecurity-Dataset` 上单卡微调 Gemma 4 E4B-it

基于魔搭 ModelScope 的 `hcnote/Cybersecurity-Dataset` 对 Gemma 4 E4B-it 做 LoRA 微调。

与 emotion 版本的主要区别：

1. **数据集**：网络安全指令微调数据集（`instruction`/`input`/`output`）。
2. **任务类型**：自由文本生成。
3. **Cell 顺序重排**：tokenizer 先于数据集加载，用真实 token 数过滤长样本。
4. **卸载 vision tower**：纯文本任务不需要图像编码器，移到 CPU 省显存。
5. **评估方式**：收集生成样本人工抽查。
6. 针对 AMD / ROCm 环境，不使用 `bitsandbytes` 4bit。

## 1. 安装依赖

In [ ]:
!uv pip install -U vllm modelscope transformers accelerate datasets trl peft scikit-learn pandas tqdm torchvision --no-cache -i https://mirrors.cloud.tencent.com/pypi/simple/ --extra-index-url https://wheels.vllm.ai/rocm/
!pip uninstall torchaudio -y 2>/dev/null; rm -rf /opt/venv/lib/python3.12/site-packages/torchaudio /opt/venv/lib/python3.12/site-packages/torchaudio-*.dist-info

## 2. 导入依赖与全局配置

In [ ]:
import os
import glob
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import Dataset, DatasetDict, load_dataset

from modelscope import snapshot_download
from modelscope.hub.snapshot_download import dataset_snapshot_download

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore")

# 减少显存碎片
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# -----------------------------
# 基础配置
# -----------------------------
MODELSCOPE_MODEL_ID = "google/gemma-4-E4B-it"
MODELSCOPE_DATASET_ID = "hcnote/Cybersecurity-Dataset"
OUTPUT_DIR = "./gemma4-it-cybersec-lora-ms-single-gpu"

TRAIN_LIMIT = 4000
VALIDATION_LIMIT = 400
TEST_LIMIT = 400
EVAL_LIMIT = 50

SEED = 42
MODEL_DTYPE = torch.bfloat16
BF16 = True
FP16 = False

# 过滤阈值：instruction+input 超过此 token 数的样本被丢弃
MAX_INPUT_TOKENS = 768

SYSTEM_PROMPT = """You are a cybersecurity expert assistant.
Answer the user's security-related questions accurately and concisely.
Provide technical details when appropriate."""

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)
os.makedirs("./datasets", exist_ok=True)

print("torch version:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.cuda.device_count():", torch.cuda.device_count())
if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("device name:", torch.cuda.get_device_name(0))

## 3. 固定随机种子

In [ ]:
def setup_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

setup_seed(SEED)

## 4. 下载模型

In [ ]:
print("Downloading model from ModelScope...")
model_dir = snapshot_download(MODELSCOPE_MODEL_ID, cache_dir="./models")
print("Downloaded model dir:", model_dir)
LOCAL_MODEL_DIR = model_dir

## 5. 加载 tokenizer（先于数据集，用于精准过滤长样本）

In [ ]:
print("Loading tokenizer from:", LOCAL_MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_DIR,
    use_fast=True,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

# chat_template 兜底
if not getattr(tokenizer, "chat_template", None):
    print("Loading chat_template.jinja from ModelScope ...")
    template_dir = snapshot_download(
        "google/gemma-4-E4B-it",
        cache_dir="./models",
        allow_file_pattern=["chat_template.jinja"],
    )
    path = os.path.join(template_dir, "chat_template.jinja")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            tokenizer.chat_template = f.read()
        print("Loaded chat_template, length =", len(tokenizer.chat_template))
else:
    print("tokenizer.chat_template already set.")

# 自检
_probe = tokenizer.apply_chat_template(
    [{"role": "system", "content": "Hi"}, {"role": "user", "content": "Hello"}],
    tokenize=False, add_generation_prompt=True,
)
print("chat_template probe:\n" + _probe[:200])

## 6. 加载数据集（用 tokenizer 精准过滤过长样本）

`hcnote/Cybersecurity-Dataset`：`instruction`/`input`/`output` 三字段，JSONL 格式，仅 train split。

这里用已加载的 tokenizer 真实 tokenize，`instruction+input` 超过 `MAX_INPUT_TOKENS` 的样本直接丢弃，
从源头避免 OOM，同时不截断保留样本的内容。

In [ ]:
print("Downloading dataset from ModelScope...")
dataset_dir = dataset_snapshot_download(MODELSCOPE_DATASET_ID, cache_dir="./datasets")
print("Downloaded dataset dir:", dataset_dir)

jsonl_files = sorted(glob.glob(os.path.join(dataset_dir, "*.jsonl")))
print(f"Found JSONL files: {jsonl_files}")

if not jsonl_files:
    raise FileNotFoundError(f"No .jsonl files found in {dataset_dir}.")

raw_dataset = load_dataset("json", data_files=jsonl_files, split="train")
print(f"Raw dataset: {raw_dataset}")
print(f"Total samples: {len(raw_dataset)}")

# 用 tokenizer 精准过滤过长样本
def not_too_long(example):
    text = example.get("instruction", "") + "\n\n" + example.get("input", "")
    tokens = tokenizer.encode(text, truncation=False)
    return len(tokens) <= MAX_INPUT_TOKENS

raw_dataset = raw_dataset.filter(not_too_long, desc="Filtering long samples")
print(f"After filtering: {len(raw_dataset)} samples (<= {MAX_INPUT_TOKENS} tokens)")
print(f"Example:\n{json.dumps(raw_dataset[0], indent=2, ensure_ascii=False)}")

# 自行划分 train / validation / test
raw_dataset = raw_dataset.shuffle(seed=SEED)

total = len(raw_dataset)
train_end = min(TRAIN_LIMIT, total)
val_end = train_end + min(VALIDATION_LIMIT, max(0, total - train_end))
test_end = val_end + min(TEST_LIMIT, max(0, total - val_end))

dataset = DatasetDict({
    "train": raw_dataset.select(range(0, train_end)),
    "validation": raw_dataset.select(range(train_end, val_end)),
    "test": raw_dataset.select(range(val_end, test_end)),
})

print(dataset)
print("train example:")
print(json.dumps(dataset["train"][0], indent=2, ensure_ascii=False))

## 7. 构造 prompt-completion 格式数据

In [ ]:
def to_prompt_completion(example):
    instruction = example.get("instruction", "")
    user_input = example.get("input", "")
    output = example.get("output", "")

    if user_input:
        user_content = f"{instruction}\n\n{user_input}"
    else:
        user_content = instruction

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "completion": [
            {"role": "assistant", "content": output},
        ],
    }


sft_dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset["train"].column_names,
)

print(sft_dataset)
print("train example:")
print(sft_dataset["train"][0])

## 8. 加载模型\n\n从本地路径加载 Gemma-4-E4B-it，BF16 单卡。\n\n如果这里 OOM，可以改 `MODEL_DTYPE = torch.float16`，或降低 batch size。

In [ ]:
print("Loading base model from:", LOCAL_MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("HIP version:", getattr(torch.version, "hip", None))

base_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

base_model.to(device)

base_model.config.use_cache = False
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.bos_token_id = tokenizer.bos_token_id
base_model.config.eos_token_id = tokenizer.eos_token_id

base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.bos_token_id = tokenizer.bos_token_id
base_model.generation_config.eos_token_id = tokenizer.eos_token_id

print("Base model loaded.")
print("Base model device:", next(base_model.parameters()).device)

## 9. 推理函数

In [ ]:
def generate_response(model, tokenizer, instruction: str, user_input: str = "",
                      system_prompt: str = SYSTEM_PROMPT, max_new_tokens: int = 256) -> str:
    if user_input:
        user_content = f"{instruction}\n\n{user_input}"
    else:
        user_content = instruction

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]

    _device = next(model.parameters()).device

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    )
    inputs = {k: v.to(_device) for k, v in inputs.items()}

    input_len = inputs["input_ids"].shape[-1]
    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


# 快速测试
test_response = generate_response(
    base_model, tokenizer,
    instruction="What is a buffer overflow attack?",
    max_new_tokens=128,
)
print("Test response:")
print(test_response[:300])

## 10. 评估函数

In [ ]:
def evaluate_model(model, tokenizer, split="test", limit=EVAL_LIMIT):
    rows = []
    raw_source = dataset[split]
    if limit is not None:
        raw_source = raw_source.select(range(min(limit, len(raw_source))))

    model.eval()
    for ex in tqdm(raw_source, desc=f"Evaluating {split}", leave=False):
        prediction = generate_response(
            model, tokenizer,
            instruction=ex.get("instruction", ""),
            user_input=ex.get("input", ""),
            max_new_tokens=256,
        )
        rows.append({
            "instruction": ex.get("instruction", ""),
            "input": ex.get("input", ""),
            "reference": ex.get("output", ""),
            "prediction": prediction,
        })
    return pd.DataFrame(rows)


def print_sample_comparison(pre_df, post_df, n=5):
    for i in range(min(n, len(pre_df))):
        print(f"{'='*60}")
        print(f"Sample {i+1}")
        print(f"Instruction: {pre_df.iloc[i]['instruction'][:200]}")
        print(f"Input: {pre_df.iloc[i]['input'][:200]}")
        print(f"\n--- Reference ---")
        print(pre_df.iloc[i]['reference'][:300])
        print(f"\n--- Pre-finetuning ---")
        print(pre_df.iloc[i]['prediction'][:300])
        print(f"\n--- Post-finetuning ---")
        print(post_df.iloc[i]['prediction'][:300])
        print()

## 11. 微调前评估

In [ ]:
pre_preds = evaluate_model(base_model, tokenizer, split="test", limit=EVAL_LIMIT)
print(f"Pre-finetuning evaluated {len(pre_preds)} samples")
pre_preds[["instruction", "reference", "prediction"]].head(5)

## 12. 配置 LoRA

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

## 13. 训练参数

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=50,
    num_train_epochs=1,

    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    gradient_checkpointing=True,

    bf16=BF16,
    fp16=FP16,
    tf32=False,

    max_length=512,
    packing=False,
    completion_only_loss=True,

    remove_unused_columns=False,
    dataloader_num_workers=2,

    optim="adamw_torch",
    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

## 14. LoRA 微调

In [ ]:
if isinstance(base_model, PeftModel):
    base_model = base_model.unload()
    base_model.config.use_cache = False

trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer,
)

trainable_params = 0
total_params = 0
trainable_param_names = []

for name, param in trainer.model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        trainable_param_names.append(name)

if trainable_params == 0:
    raise RuntimeError("No trainable LoRA parameters. Check target_modules.")

print(f"Trainable LoRA parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable ratio: {100 * trainable_params / total_params:.4f}%")
print("Example trainable parameters:")
print(trainable_param_names[:20])

train_result = trainer.train()

trainer.model.eval()
trainer.model.config.use_cache = True
train_result

## 15. 保存 LoRA adapter

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

print("Saved adapter and tokenizer to:", OUTPUT_DIR)

## 16. 微调后评估

In [ ]:
ft_model = trainer.model
ft_model.eval()
ft_model.config.use_cache = True

post_preds = evaluate_model(ft_model, tokenizer, split="test", limit=EVAL_LIMIT)
print(f"Post-finetuning evaluated {len(post_preds)} samples")
post_preds[["instruction", "reference", "prediction"]].head(5)

## 17. 微调前后对比

In [ ]:
print_sample_comparison(pre_preds, post_preds, n=5)

## 18. 手动测试

In [ ]:
def predict_cybersec(instruction: str, user_input: str = "") -> str:
    return generate_response(ft_model, tokenizer, instruction, user_input, max_new_tokens=256)


test_queries = [
    ("What is a buffer overflow attack and how can it be prevented?", ""),
    ("How to detect a SQL injection vulnerability in a web application?", ""),
    ("Explain the difference between XSS and CSRF attacks.", ""),
    ("What is a zero-day vulnerability?", ""),
    ("Describe best practices for securing a Linux server.", ""),
]

for instruction, user_input in test_queries:
    response = predict_cybersec(instruction, user_input)
    print(f"Q: {instruction}")
    print(f"A: {response[:400]}")
    print("---")

## 19. 保存结果

In [ ]:
pre_preds.to_csv(os.path.join(OUTPUT_DIR, "pre_finetuning_predictions.csv"), index=False)
post_preds.to_csv(os.path.join(OUTPUT_DIR, "post_finetuning_predictions.csv"), index=False)

comparison = pre_preds[["instruction", "input", "reference"]].copy()
comparison["pre_prediction"] = pre_preds["prediction"]
comparison["post_prediction"] = post_preds["prediction"]
comparison.to_csv(os.path.join(OUTPUT_DIR, "gemma4_cybersec_before_after_comparison.csv"), index=False)

print("Saved all outputs to:", OUTPUT_DIR)

## 20. 可选：重新加载 LoRA adapter 推理

In [ ]:
# 可选运行：重新加载 LoRA adapter
# 为避免当前 notebook 占用显存，建议重启 kernel 后单独运行本节。

RUN_RELOAD_TEST = False

if RUN_RELOAD_TEST:
    reload_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, use_fast=True, trust_remote_code=True)
    if reload_tokenizer.pad_token is None:
        reload_tokenizer.pad_token = reload_tokenizer.eos_token

    reload_base_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_DIR, torch_dtype=MODEL_DTYPE, low_cpu_mem_usage=True, trust_remote_code=True,
    )
    reload_model = PeftModel.from_pretrained(reload_base_model, OUTPUT_DIR)
    reload_model.eval()

    print(generate_response(
        reload_model, reload_tokenizer,
        instruction="What is a buffer overflow attack?",
        max_new_tokens=256,
    ))

## 21. 威胁情报分析面板（复用已加载模型）

In [ ]:
# 复用当前 notebook 已加载的 ft_model 和 tokenizer，不重新加载模型

SYSTEM_PROMPT_INTEL = """You are a threat intelligence analyst. Output a JSON report with this exact schema:
{
  "summary": "one paragraph threat summary",
  "risk_level": "critical|high|medium|low|info",
  "ttps": [{"tactic": "MITRE ATT&CK tactic", "technique_id": "TXXXX", "technique_name": "name", "description": "how it applies"}],
  "iocs": [{"type": "ip|domain|url|hash_md5|hash_sha256", "value": "...", "confidence": "high|medium|low"}],
  "affected_systems": ["system names"],
  "recommendations": ["actionable steps"],
  "related_threats": ["threat actor, campaign, or malware family"]
}
Output ONLY the JSON object, no other text."""


class ThreatIntelQuick:
    """轻量级威胁情报分析器 — 复用已加载的微调模型"""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device

    def _generate(self, instruction: str, user_input: str) -> str:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_INTEL},
            {"role": "user", "content": f"{instruction}\n\n{user_input}"},
        ]
        inputs = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=512, do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        return self.tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    def analyze_ioc(self, ioc_value: str):
        """分析单个 IOC（IP/域名/哈希/URL）"""
        return self._generate(
            "Analyze this IOC. What threat does it indicate? Map to MITRE ATT&CK:",
            ioc_value,
        )

    def analyze_cve(self, cve_id: str, description: str = ""):
        """分析 CVE 漏洞"""
        text = f"CVE: {cve_id}"
        if description:
            text += f"\nDescription: {description}"
        return self._generate("Analyze this CVE. Assess impact and recommend mitigations:", text)

    def analyze_raw(self, text: str):
        """通用威胁情报分析"""
        return self._generate(
            "Analyze this threat intelligence. Extract IOCs, map TTPs, assess risk, recommend actions:",
            text[:3000],
        )

    def _parse_and_display(self, raw: str):
        """解析 JSON 输出并格式化显示"""
        import re as _re
        match = _re.search(r'\{[\s\S]*\}', raw)
        if match:
            try:
                data = json.loads(match.group())
                print(f"\n{'='*50}")
                print(f"  Risk: {data.get('risk_level', '?').upper()}")
                print(f"  Summary: {data.get('summary', '')[:300]}")
                if data.get('ttps'):
                    print(f"\n  MITRE ATT&CK Mapping:")
                    for t in data['ttps'][:8]:
                        print(f"    [{t.get('technique_id', '?')}] {t.get('technique_name', '?')} → {t.get('tactic', '?')}")
                if data.get('iocs'):
                    print(f"\n  Extracted IOCs:")
                    for ioc in data['iocs'][:8]:
                        print(f"    {ioc.get('type', '?')}: {ioc.get('value', '?')} ({ioc.get('confidence', '?')})")
                if data.get('affected_systems'):
                    print(f"\n  Affected: {', '.join(data['affected_systems'][:5])}")
                if data.get('recommendations'):
                    print(f"\n  Recommendations:")
                    for i, r in enumerate(data['recommendations'][:5], 1):
                        print(f"    {i}. {r}")
                if data.get('related_threats'):
                    print(f"\n  Related Threats: {', '.join(data['related_threats'][:5])}")
                print(f"{'='*50}")
                return data
            except json.JSONDecodeError:
                pass
        print(f"\n{'='*50}")
        print(raw[:800])
        print(f"{'='*50}")

    def run_panel(self):
        """交互式威胁情报面板"""
        print("""
╔══════════════════════════════════════════╗
║    🔴 AI 威胁情报分析面板 (Quick)        ║
║    Gemma-4-E4B-it + Cybersecurity LoRA   ║
╠══════════════════════════════════════════╣
║  [IOC] IP/域名/哈希/URL                 ║
║  [CVE] CVE-XXXX-XXXX                    ║
║  [RAW] 粘贴任意情报文本                   ║
║  [q]   退出                              ║
╚══════════════════════════════════════════╝
""")
        while True:
            try:
                cmd = input("\n[panel] ▶ ").strip()
            except (EOFError, KeyboardInterrupt):
                print("\nExiting.")
                break

            if cmd.lower() in ('q', 'quit', 'exit'):
                break
            elif cmd.upper().startswith('CVE'):
                desc = input("  Description (optional): ").strip()
                print("  [*] Analyzing...")
                raw = self.analyze_cve(cmd, desc)
                self._parse_and_display(raw)
            elif cmd.upper() in ('IOC', 'IP', 'DOMAIN', 'HASH', 'URL'):
                val = input("  Value: ").strip()
                if val:
                    print("  [*] Analyzing...")
                    raw = self.analyze_ioc(val)
                    self._parse_and_display(raw)
            elif cmd.upper() == 'RAW':
                print("  Paste intelligence text (empty line to submit):")
                lines = []
                while True:
                    try:
                        line = input()
                    except EOFError:
                        break
                    if not line:
                        break
                    lines.append(line)
                if lines:
                    print("  [*] Analyzing...")
                    raw = self.analyze_raw('\n'.join(lines))
                    self._parse_and_display(raw)
            elif cmd:
                # 直接当 IOC 或自由文本处理
                print("  [*] Analyzing...")
                raw = self.analyze_raw(cmd)
                self._parse_and_display(raw)


# 初始化（复用 ft_model）
intel = ThreatIntelQuick(ft_model, tokenizer)
print("ThreatIntelQuick initialized. Run intel.run_panel() to start.")
print('Quick test: intel.analyze_ioc("CVE-2024-3094") or intel.run_panel()')

## 常见问题

### OOM 怎么办？

1. 降低 `MAX_INPUT_TOKENS`（如 512）
2. 降低 `TRAIN_LIMIT` 和 `EVAL_LIMIT`
3. 降低 `per_device_train_batch_size` 到 1 或 2
4. 降低 `max_length` 到 384

### 显存优化说明

本 notebook 已启用两项优化：
- **vision tower 卸载到 CPU**：纯文本任务不需要，省 2-3GB
- **`PYTORCH_ALLOC_CONF=expandable_segments:True`**：减少显存碎片
- **tokenizer 精准过滤**：`instruction+input` 超过 `MAX_INPUT_TOKENS` 的样本在源头丢弃